### Importing Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics._regression import mean_absolute_error, mean_squared_error, root_mean_squared_error


In [2]:
os.listdir( '../data/preprocessed data' )

['pre-processed_data.csv']

In [3]:
df = pd.read_csv( '../data/preprocessed data/pre-processed_data.csv' )

In [4]:
df.sample()

,study_hours,class_attendance,sleep_hours,sleep_quality,facility_rating,exam_difficulty,exam_score,course_diff,gender_female,gender_male,...,course_Bachelor of Science,course_Bachelor of Technology,course_Diploma,internet_access_no,internet_access_yes,study_method_coaching,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study
19231,0.29,78.1,8.4,1,1,0,51.6,3,0,1,...,0,0,0,0,1,1,0,0,0,0


## Split features into X & y

In [5]:
X = df.drop( columns = 'exam_score' )
y = df['exam_score']

In [6]:
X.shape

(20000, 24)

## Split into training & testing

In [7]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# Model Selection

### Ensemble models

In [8]:
knn_model = KNeighborsRegressor( n_neighbors= 1 )

In [9]:
X_train.shape

(16000, 24)

In [10]:
X_test.shape

(4000, 24)

In [11]:
y_test

10650     31.1
2041      81.6
8668      68.0
1114     100.0
13902     84.8
         ...  
4073      58.6
7442      53.3
9999      58.8
1870      45.3
15196     42.7
Name: exam_score, Length: 4000, dtype: float64

In [12]:
knn_model.fit( X_train, y_train )

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric metric: str, DistanceMetric object or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.If metric is a DistanceMetric object, it will be passed directly tothe underlying computation routines.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [13]:
y_pred_train = knn_model.predict( X_train )
y_pred_train[:5]

array([87.6, 80.4, 94.5, 50.7, 79.8])

In [14]:
y_test[100:106]

15304    79.0
4309     47.5
4484     58.3
11013    75.1
4770     47.6
2712     49.2
Name: exam_score, dtype: float64

In [15]:
18 / 3

6.0

In [16]:
y_pred_test = knn_model.predict( X_test )

In [17]:
y_pred_test[100:106]

array([79.5, 32.2, 52. , 70.2, 56.3, 42.3])

In [18]:
pd.DataFrame({
    'Actual' : y_test[100:106],
    'Predicted' : y_pred_test[100:106],
    'Residuals' : y_test[100:106] - y_pred_test[100:106]
})

,Actual,Predicted,Residuals
15304,79.0,79.5,-0.5
4309,47.5,32.2,15.3
4484,58.3,52.0,6.3
11013,75.1,70.2,4.9
4770,47.6,56.3,-8.7
2712,49.2,42.3,6.9


In [19]:
0.5 + 15.3 + 6.3 + 4.9 

27.0

## metrics

### MAE ( Mean Absolute Error ) ->  Look on the Avg. Error
### MSE ( Mean Squared Error )  ->  Triggers if there is high residuals (differences)
### RMSE ( Root Mean Squared Error ) -> Error between MAE & MSE

In [20]:
MAE_train = mean_absolute_error( y_train, y_pred_train )
MAE_test = mean_absolute_error( y_test, y_pred_test )


MSE_train = mean_squared_error( y_train, y_pred_train )
MSE_test = mean_squared_error( y_test, y_pred_test )

RMSE_train = root_mean_squared_error( y_train, y_pred_train )
RMSE_test = root_mean_squared_error( y_test, y_pred_test )



print( f'Mean Absolute Error of training : { MAE_train }' )
print( f'Mean Absolute Error of Testing : { MAE_test }' )

print(  )
print( f'Mean Squared Error of training : { MSE_train }' )
print( f'Mean Squared Error of Testing : { MSE_test }' )
print(  )
print( f'Root Mean Squared Error of Training : { RMSE_train }' )
print( f'Root Mean Squared Error of Testing : { RMSE_test }' )


Mean Absolute Error of training : 0.0
Mean Absolute Error of Testing : 12.216742750000002

Mean Squared Error of training : 0.0
Mean Squared Error of Testing : 234.81223721774998

Root Mean Squared Error of Training : 0.0
Root Mean Squared Error of Testing : 15.323584346286282


### Evaluation of previous n_neighbors = 5
**9.4, 136, 11.6**

### Calc. Cross Validation Score

In [21]:
cross_val_score( knn_model, X_train, y_train, cv= 4, scoring='neg_mean_squared_error' ).mean()

np.float64(-232.82450976693747)

In [22]:
pd.set_option( 'display.max_columns', None )

In [23]:
X_train.sample()

,study_hours,class_attendance,sleep_hours,sleep_quality,facility_rating,exam_difficulty,course_diff,gender_female,gender_male,gender_other,course_Bachelor of Arts,course_Bachelor of Business Administration,course_Bachelor of Commerce,course_Bachelor of Computer Applications,course_Bachelor of Science,course_Bachelor of Technology,course_Diploma,internet_access_no,internet_access_yes,study_method_coaching,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study
1425,1.4,81.0,5.3,1,1,1,1,1,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0


In [24]:
234.81223721774998 ** 0.5

15.323584346286282

## Curse of Dimensiality

## Fine Tuning

# Testing

In [25]:
df.sample()

,study_hours,class_attendance,sleep_hours,sleep_quality,facility_rating,exam_difficulty,exam_score,course_diff,gender_female,gender_male,gender_other,course_Bachelor of Arts,course_Bachelor of Business Administration,course_Bachelor of Commerce,course_Bachelor of Computer Applications,course_Bachelor of Science,course_Bachelor of Technology,course_Diploma,internet_access_no,internet_access_yes,study_method_coaching,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study
1444,0.7,68.3,8.3,1,0,1,32.7,6,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0


## Testing GridSearchCV on KNN Model

In [26]:
knn_model = KNeighborsRegressor(  )

In [27]:
gs_knn =  GridSearchCV(
    estimator= knn_model,
    param_grid= { 'n_neighbors' : [ 2, 3, 4, 5, 6, 7 ] }
)

In [28]:
gs_knn.fit( X_train, y_train )

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'n_neighbors': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter 

In [29]:
gs_knn.best_params_

{'n_neighbors': 7}

In [30]:
y_test

10650     31.1
2041      81.6
8668      68.0
1114     100.0
13902     84.8
         ...  
4073      58.6
7442      53.3
9999      58.8
1870      45.3
15196     42.7
Name: exam_score, Length: 4000, dtype: float64

In [31]:
y_pred_test

array([45.7, 51.5, 67.4, ..., 57.4, 63.9, 63.2], shape=(4000,))

In [32]:
knn_comparison_df = pd.DataFrame({
    'Actual (True)' : y_test,
    'Predicted (KNN)' : y_pred_test,
    'Residuals' : np.abs( y_test- y_pred_test )
})
knn_comparison_df

,Actual (True),Predicted (KNN),Residuals
10650,31.1,45.7,14.6
2041,81.6,51.5,30.1
8668,68.0,67.4,0.6
1114,100.0,100.0,0.0
13902,84.8,92.8,8.0
...,...,...,...
4073,58.6,60.6,2.0
7442,53.3,48.4,4.9
9999,58.8,57.4,1.4
1870,45.3,63.9,18.6


In [33]:
knn_comparison_df

,Actual (True),Predicted (KNN),Residuals
10650,31.1,45.7,14.6
2041,81.6,51.5,30.1
8668,68.0,67.4,0.6
1114,100.0,100.0,0.0
13902,84.8,92.8,8.0
...,...,...,...
4073,58.6,60.6,2.0
7442,53.3,48.4,4.9
9999,58.8,57.4,1.4
1870,45.3,63.9,18.6


## Visulauize Residuals

In [34]:
px.line( knn_comparison_df.head( 25 ), x  = ['Actual (True)', 'Predicted (KNN)'] )

In [35]:
knn_comparison_df.head( 25 )

,Actual (True),Predicted (KNN),Residuals
10650,31.1,45.700,14.600
2041,81.6,51.500,30.100
8668,68.0,67.400,0.600
1114,100.0,100.000,0.000
13902,84.8,92.800,8.000
11963,52.8,43.900,8.900
11072,53.1,50.200,2.900
3002,81.0,98.800,17.800
19771,72.6,50.400,22.200
8115,25.4,19.599,5.801


In [36]:
px.histogram( knn_comparison_df, x  = ['Actual (True)', 'Predicted (KNN)'] )

<hr>

# [2] Linear Regression

In [37]:
from sklearn.linear_model import LinearRegression

In [38]:
lin_reg_model = LinearRegression()

In [39]:
os.cpu_count()

4

In [40]:
lin_reg_model.fit( X_train, y_train )

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [41]:
y_pred_train_lr = lin_reg_model.predict( X_train )
y_pred_test_lr = lin_reg_model.predict( X_test )

In [42]:
y_pred_train_lr

array([77.08540499, 83.54623209, 80.63310689, ..., 57.29706372,
       40.55708554, 51.19058777], shape=(16000,))

In [43]:
y_train

5894     87.6
3728     80.4
8958     94.5
7671     50.7
5999     79.8
         ... 
11284    75.4
11964    83.8
5390     59.5
860      44.0
15795    49.7
Name: exam_score, Length: 16000, dtype: float64

# Intercept -> b

In [44]:
lin_reg_model.intercept_

np.float64(-3.8845610110956343)

In [45]:
X_train.shape

(16000, 24)

In [46]:
lin_reg_model.coef_

array([ 5.87984532e+00,  3.44686431e-01,  1.43688262e+00,  4.66803584e+00,
        3.90089950e+00,  7.57607919e-02, -5.27043041e-03, -8.23191561e-02,
       -3.45641438e-02,  1.16883300e-01,  5.06988542e-02, -3.55331817e-02,
       -3.71061723e-02,  8.79725266e-02, -2.28248471e-01,  1.52365809e-01,
        9.85063512e-03, -3.91517725e-02,  3.91517725e-02,  6.22185033e+00,
       -1.55578786e+00,  1.24665746e+00, -2.64984969e+00, -3.26287025e+00])

### Optimization Algorithm

In [47]:
X_train.columns

Index(['study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality',
       'facility_rating', 'exam_difficulty', 'course_diff', 'gender_female',
       'gender_male', 'gender_other', 'course_Bachelor of Arts',
       'course_Bachelor of Business Administration',
       'course_Bachelor of Commerce',
       'course_Bachelor of Computer Applications',
       'course_Bachelor of Science', 'course_Bachelor of Technology',
       'course_Diploma', 'internet_access_no', 'internet_access_yes',
       'study_method_coaching', 'study_method_group study',
       'study_method_mixed', 'study_method_online videos',
       'study_method_self-study'],
      dtype='str')

## Y = b + p1x1 + p2x2 + p3x3 + p4x4 +  .......... p24x24

In [48]:
print( 'Linear Regression Evaluation :' )
print( '-------------------------' )
print(  )

MAE_train = mean_absolute_error( y_train, y_pred_train_lr )
MAE_test = mean_absolute_error( y_test, y_pred_test_lr )


MSE_train = mean_squared_error( y_train, y_pred_train_lr )
MSE_test = mean_squared_error( y_test, y_pred_test_lr )

RMSE_train = root_mean_squared_error( y_train, y_pred_train_lr )
RMSE_test = root_mean_squared_error( y_test, y_pred_test_lr )

print( f'Mean Absolute Error of training : { MAE_train }' )
print( f'Mean Absolute Error of Testing : { MAE_test }' )

print(  )
print( f'Mean Squared Error of training : { MSE_train }' )
print( f'Mean Squared Error of Testing : { MSE_test }' )
print(  )
print( f'Root Mean Squared Error of Training : { RMSE_train }' )
print( f'Root Mean Squared Error of Testing : { RMSE_test }' )

Linear Regression Evaluation :
-------------------------

Mean Absolute Error of training : 7.849729059806823
Mean Absolute Error of Testing : 7.86201359973347

Mean Squared Error of training : 95.82557389270734
Mean Squared Error of Testing : 95.46814634090096

Root Mean Squared Error of Training : 9.789053779232564
Root Mean Squared Error of Testing : 9.770780231941611


### Model Serialization ( Saving ) ( Dumping )
#### Save to extension .pkl

In [49]:
import joblib
import pickle

In [50]:
lin_reg_model

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [51]:
os.listdir( '../artifacts' )

[]

In [52]:
# joblib.dump( lin_reg_model, '../artifacts/exam_score_model.pkl' )

In [53]:
df.sample()

,study_hours,class_attendance,sleep_hours,sleep_quality,facility_rating,exam_difficulty,exam_score,course_diff,gender_female,gender_male,gender_other,course_Bachelor of Arts,course_Bachelor of Business Administration,course_Bachelor of Commerce,course_Bachelor of Computer Applications,course_Bachelor of Science,course_Bachelor of Technology,course_Diploma,internet_access_no,internet_access_yes,study_method_coaching,study_method_group study,study_method_mixed,study_method_online videos,study_method_self-study
7174,7.14,48.2,6.8,1,0,2,67.6,6,0,1,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0


In [54]:
df.shape

(20000, 25)

### Define a function to evaluate any regressor

In [55]:
#### model

In [56]:
from colorama import Fore

In [57]:
print(  Fore.MAGENTA + 'hello' )

hello


In [58]:
def evaluate_regressor( model, model_name: str ,X_train, X_test, y_train, y_test  ) :
    
    print( 'Training.........' + '\n' )
    
    print( model_name + ':' )
    print( '-'*20 )
    
    model.fit( X_train, y_train ) # Training !
    y_pred_train = model.predict( X_train )
    y_pred_test = model.predict( X_test )

    print(  )

    MAE_train = mean_absolute_error( y_train, y_pred_train )
    MSE_train = mean_squared_error( y_train, y_pred_train )
    RMSE_train = root_mean_squared_error( y_train, y_pred_train )
    
    MAE_test = mean_absolute_error( y_test, y_pred_test )
    MSE_test = mean_squared_error( y_test, y_pred_test )
    RMSE_test = root_mean_squared_error( y_test, y_pred_test )

    print( f'Mean Absolute Error of training : { MAE_train }' )
    print( Fore.CYAN + f'Mean Squared Error of training : { MSE_train }' )
    print( Fore.WHITE + f'Root Mean Squared Error of Training : { RMSE_train }' )
    print(  )

    print( Fore.WHITE + f'Mean Absolute Error of Testing : { MAE_test }' )
    print( Fore.RED  + f'Mean Squared Error of Testing : { MSE_test }' )
    print( Fore.WHITE + f'Root Mean Squared Error of Testing : { RMSE_test }' )


## [1] KNN

In [59]:
knn_model = KNeighborsRegressor( n_neighbors=7 )

In [60]:
evaluate_regressor( knn_model, 'K Neighbors Regressor', X_train, X_test, y_train, y_test )

Training.........

K Neighbors Regressor:
--------------------

Mean Absolute Error of training : 7.819387026785715
Mean Squared Error of training : 94.85337002898342
Root Mean Squared Error of Training : 9.73926948127956

Mean Absolute Error of Testing : 9.272969892857141
Mean Squared Error of Testing : 131.7767803332194
Root Mean Squared Error of Testing : 11.47940679361174


## [2] Linear Regression 

In [61]:
lin_reg_model = LinearRegression()

In [62]:
evaluate_regressor( lin_reg_model, 'Linear Regression without regularization', X_train, X_test, y_train, y_test  )

Training.........

Linear Regression without regularization:
--------------------

Mean Absolute Error of training : 7.849729059806823
Mean Squared Error of training : 95.82557389270734
Root Mean Squared Error of Training : 9.789053779232564

Mean Absolute Error of Testing : 7.86201359973347
Mean Squared Error of Testing : 95.46814634090096
Root Mean Squared Error of Testing : 9.770780231941611


## [3] Linear Regression with regularization L1 ( Lasso )

In [63]:
lin_reg_L1 = Lasso( alpha = 0.2 )

In [64]:
evaluate_regressor( lin_reg_L1, 'Linear Regression with REGULARIZATION ( L1 )', X_train, X_test, y_train, y_test  )

Training.........

Linear Regression with REGULARIZATION ( L1 ):
--------------------

Mean Absolute Error of training : 7.896820765859728
Mean Squared Error of training : 96.78491626380304
Root Mean Squared Error of Training : 9.837932519782957

Mean Absolute Error of Testing : 7.918284948378739
Mean Squared Error of Testing : 96.60217510369358
Root Mean Squared Error of Testing : 9.82864055216659


## [4] Linear Regression with regularization L2 ( Ridge )

In [65]:
ridge = Ridge(  )

In [66]:
evaluate_regressor( ridge, 'Linear Regression with REGULARIZATION ( L2 )', X_train, X_test, y_train, y_test  )

Training.........

Linear Regression with REGULARIZATION ( L2 ):
--------------------

Mean Absolute Error of training : 7.849739744732457
Mean Squared Error of training : 95.82557531103602
Root Mean Squared Error of Training : 9.78905385167719

Mean Absolute Error of Testing : 7.862034248342009
Mean Squared Error of Testing : 95.46824428283712
Root Mean Squared Error of Testing : 9.770785243921653


## [5] Support Vector Machine

In [67]:
from sklearn.svm import SVR

In [68]:
svm_regressor = SVR(  )

In [69]:
evaluate_regressor( svm_regressor, 'SVM', X_train, X_test, y_train, y_test  )

Training.........

SVM:
--------------------

Mean Absolute Error of training : 9.628362006255745
Mean Squared Error of training : 142.56961963495658
Root Mean Squared Error of Training : 11.940252075854872

Mean Absolute Error of Testing : 9.809015763770596
Mean Squared Error of Testing : 145.43736840460892
Root Mean Squared Error of Testing : 12.059741639214703


## [6] Support Vector Machine with Sacling (SVM)

In [70]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [71]:
svm_pipeline =   make_pipeline(
    StandardScaler(),
    svm_regressor
)

In [72]:
svm_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


In [73]:
evaluate_regressor( svm_pipeline, 'SVM With Scaling', X_train, X_test, y_train, y_test  )

Training.........

SVM With Scaling:
--------------------

Mean Absolute Error of training : 7.764041233140608
Mean Squared Error of training : 95.38082144931604
Root Mean Squared Error of Training : 9.76631053414318

Mean Absolute Error of Testing : 8.02693714893925
Mean Squared Error of Testing : 98.9454700445467
Root Mean Squared Error of Testing : 9.947133760262133


<hr>

In [74]:
df.columns

Index(['study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality',
       'facility_rating', 'exam_difficulty', 'exam_score', 'course_diff',
       'gender_female', 'gender_male', 'gender_other',
       'course_Bachelor of Arts', 'course_Bachelor of Business Administration',
       'course_Bachelor of Commerce',
       'course_Bachelor of Computer Applications',
       'course_Bachelor of Science', 'course_Bachelor of Technology',
       'course_Diploma', 'internet_access_no', 'internet_access_yes',
       'study_method_coaching', 'study_method_group study',
       'study_method_mixed', 'study_method_online videos',
       'study_method_self-study'],
      dtype='str')

# Session 21
# Try Fine Tuning
- if kernel == 'poly' --> higher degree --> pruning overfiting
- if gamma small --> try mot to fit all data --> high error --> underfitting
- if gamma high --> fit all the data as possible --> low error --> overfitting

In [ ]:
SVR()

# Es gibt verschiedene Kernel Modele:  ['linear', 'poly', 'rbf', 'sigmoid', 'precomputed']
# Parameters: degree, gamma, coef0, tol, C, epsilon, shrinking, 

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


# C
- C --> regularization parameter
-  C --> model tries hard to fit all data --> risk overfitting
- C --> model allows erros --> risk underfitting

In [78]:
svm_model = SVR()

- zb gamma 0.2 x C 0.2
- zb gamma 0.2 x C 0.3
- zb gamma 0.2 x C 0.4 bis am 0.7
#### Dann C x gamma
- zb C 0.2 x gamma 0.2
- zb C 0.2 x gamma 0.3
- zb C 0.2 x gamma 0.4 bis am 0.7

In [79]:
gs_svm = GridSearchCV(
    svm_regressor,
    param_grid= {
        'gamma' : [0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
        'C' :[0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
    }
)

In [ ]:
# gs_svm.fit(X_train, y_train)

# braucht sehr lange Zeit !!

KeyboardInterrupt: 

# Model Serialization ( Saving ) ( dumping)

In [81]:
svm_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


In [82]:
import joblib
joblib.dump( svm_pipeline, '../artifacts/svm_pipeline.pkl')

['../artifacts/svm_pipeline.pkl']

In [ ]:
df["exam_score"]

exam_score
100.000    493
19.599     200
69.400      54
61.300      54
57.600      52
          ... 
22.800       2
20.500       2
20.100       2
22.400       2
21.900       1
Name: count, Length: 805, dtype: int64